# Phase 4 - Notebook 08: DepthSplat & 2025 Advances

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase4/08_depthsplat_2025_advances.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand how **monocular depth priors** (DPT, Depth Anything) enhance feed-forward 3DGS
2. Master the **DepthSplat** architecture and its depth-prior integration strategy
3. Survey **Splatt3R, Flash3D** and single-image 3D reconstruction methods
4. Understand **multi-view scaling** challenges (from 2 to N views)
5. Explore **2025 trends**: video-based, dynamic scenes, and language-guided 3DGS

**Estimated Time**: 60 minutes

**Prerequisites**: Notebooks 03 (MVSplat Architecture), 04 (pixelSplat), 07 (Inference & Evaluation)

---

In [ ]:
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print("\nNotebook 08: DepthSplat & 2025 Advances")

## 1. The Depth Prior Revolution

### 1.1 Why Depth Priors Matter

MVSplat and pixelSplat predict depth from **scratch** using only multi-view cues. But monocular depth estimation has made enormous progress:

| Model | Year | Key Innovation | Relative Depth Quality |
|-------|------|----------------|------------------------|
| MiDaS | 2020 | Multi-dataset training | Good |
| DPT | 2021 | ViT backbone for depth | Very Good |
| Depth Anything V1 | 2024 | Unlabeled data at scale | Excellent |
| Depth Anything V2 | 2024 | Synthetic + real mix | State-of-the-art |
| Metric3D v2 | 2024 | Metric depth (not just relative) | SOTA metric |

**Key insight**: These models produce high-quality **monocular depth** that provides a strong geometric prior, even without multi-view correspondence.

### 1.2 The Problem with Monocular Depth

Monocular depth is powerful but has fundamental limitations:

1. **Scale ambiguity**: Relative depth only (no metric scale)
2. **Shift ambiguity**: Offset is arbitrary
3. **Multi-view inconsistency**: Each view produces independent depth
4. **Edge artifacts**: Depth boundaries often don't align with image edges

**DepthSplat's key contribution**: Combining monocular depth priors with multi-view cost volumes to get the best of both worlds.

In [ ]:
# Visualize why monocular depth alone is insufficient

fig, axes = plt.subplots(2, 4, figsize=(20, 9))

# Row 1: Monocular depth strengths
np.random.seed(42)
H, W = 64, 64

# Simulated scene depth (ground truth)
y, x = np.mgrid[0:H, 0:W].astype(float)
gt_depth = 2.0 + 0.02 * x + 0.5 * np.exp(-((x-32)**2 + (y-32)**2) / 200)

# Mono depth: good shape, wrong scale
mono_depth = (gt_depth - gt_depth.min()) / (gt_depth.max() - gt_depth.min())
mono_depth = mono_depth * 3.0 + 1.0  # Different scale

# MVS depth: noisy but correct scale
mvs_depth = gt_depth + np.random.randn(H, W) * 0.3

# Combined (DepthSplat-style): best of both
combined_depth = gt_depth + np.random.randn(H, W) * 0.08

# Synthetic RGB image
rgb = np.stack([
    0.5 + 0.3 * np.sin(x * 0.2),
    0.5 + 0.3 * np.cos(y * 0.15),
    0.5 + 0.2 * np.sin((x + y) * 0.1),
], axis=-1).clip(0, 1)

# Plot
titles_top = ['Input Image', 'Ground Truth Depth', 'Monocular Depth\n(good shape, wrong scale)',
              'MVS Depth\n(correct scale, noisy)']
data_top = [rgb, gt_depth, mono_depth, mvs_depth]

for i, (title, data) in enumerate(zip(titles_top, data_top)):
    ax = axes[0, i]
    if i == 0:
        ax.imshow(data)
    else:
        im = ax.imshow(data, cmap='plasma')
        plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.axis('off')

# Row 2: Error analysis
ax = axes[1, 0]
ax.text(0.5, 0.5, 'DepthSplat\nCombined\nResult',
        transform=ax.transAxes, ha='center', va='center',
        fontsize=14, fontweight='bold', color='#2196F3')
ax.axis('off')

ax = axes[1, 1]
im = ax.imshow(combined_depth, cmap='plasma')
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title('Combined Depth\n(correct scale + clean)', fontsize=10, fontweight='bold',
             color='#2196F3')
ax.axis('off')

# Error comparison
errors = {
    'Mono (scale-aligned)': np.abs(mono_depth * (gt_depth.mean()/mono_depth.mean()) - gt_depth).mean(),
    'MVS only': np.abs(mvs_depth - gt_depth).mean(),
    'Combined (DepthSplat)': np.abs(combined_depth - gt_depth).mean(),
}

ax = axes[1, 2]
colors = ['#FF9800', '#F44336', '#4CAF50']
bars = ax.bar(list(errors.keys()), list(errors.values()), color=colors,
              edgecolor='black', lw=0.5)
ax.set_ylabel('Mean Abs Error')
ax.set_title('Depth Error Comparison', fontsize=10, fontweight='bold')
ax.tick_params(axis='x', labelsize=8)
for bar, val in zip(bars, errors.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# Cross-section comparison
ax = axes[1, 3]
mid_row = H // 2
ax.plot(gt_depth[mid_row], 'k-', lw=2, label='Ground Truth')
ax.plot(mono_depth[mid_row] * (gt_depth.mean()/mono_depth.mean()), '--',
        color='#FF9800', lw=1.5, label='Mono (scaled)')
ax.plot(mvs_depth[mid_row], '--', color='#F44336', lw=1.5, alpha=0.7, label='MVS')
ax.plot(combined_depth[mid_row], '-', color='#4CAF50', lw=1.5, label='Combined')
ax.set_xlabel('Pixel x'); ax.set_ylabel('Depth')
ax.set_title('Depth Cross-Section (middle row)', fontsize=10, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('Why Depth Priors + MVS = Better Depth', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. DepthSplat Architecture

### 2.1 Overview

**DepthSplat** (arxiv 2412.18010) enhances MVSplat by integrating monocular depth priors:

```
Input Images (N views)
    ├─── Feature Encoder (shared CNN/ViT)
    │        ↓
    │    Multi-scale Features
    │
    ├─── Depth Prior Network (DPT / Depth Anything V2)
    │        ↓
    │    Monocular Depth Maps (per view)
    │        ↓
    │    Depth-guided Plane Sampling
    │
    └─── Cost Volume Builder
             ↓
         Depth-conditioned Cost Volume
             ↓
         3D U-Net Regularization
             ↓
         Refined Depth + Gaussian Parameters
             ↓
         Pixel-aligned Gaussians
             ↓
         Differentiable Rendering
```

### 2.2 Key Innovation: Depth-guided Plane Sampling

Instead of uniform or log-uniform depth planes, DepthSplat uses the monocular depth prior to **focus depth hypotheses** around the predicted depth:

- **MVSplat**: Fixed planes in [d_min, d_max] (many wasted planes)
- **DepthSplat**: Adaptive planes centered around mono depth (all planes useful)

In [ ]:
class DepthPriorEncoder(nn.Module):
    """
    Simplified monocular depth prior encoder.
    
    In practice, this would be a pretrained DPT or Depth Anything V2 model.
    Here we use a simple CNN to demonstrate the concept.
    """
    def __init__(self, in_channels=3, feature_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.depth_head = nn.Conv2d(32, 1, 1)
        self.feature_head = nn.Conv2d(32, feature_dim, 1)
    
    def forward(self, image):
        features = self.encoder(image)
        depth = F.softplus(self.depth_head(features))  # Positive depth
        depth_features = self.feature_head(features)
        return depth, depth_features


class DepthGuidedPlaneSampler(nn.Module):
    """
    Sample depth planes adaptively around monocular depth prediction.
    
    Key idea: Instead of fixed planes in [d_min, d_max],
    sample planes around the predicted depth with adaptive range.
    """
    def __init__(self, num_planes=32, spread=0.5):
        super().__init__()
        self.num_planes = num_planes
        self.spread = spread  # How far to spread planes around prior
    
    def forward(self, mono_depth):
        """
        Args:
            mono_depth: [B, 1, H, W] monocular depth prediction
        Returns:
            depth_planes: [B, D, H, W] per-pixel depth hypotheses
        """
        B, _, H, W = mono_depth.shape
        
        # Create offsets: [-spread, +spread] around mono depth
        offsets = torch.linspace(-self.spread, self.spread, self.num_planes)
        offsets = offsets.view(1, -1, 1, 1).to(mono_depth.device)  # [1, D, 1, 1]
        
        # Scale offsets by local depth (larger depth = larger spread)
        depth_planes = mono_depth + offsets * mono_depth  # [B, D, H, W]
        depth_planes = depth_planes.clamp(min=0.1)  # Ensure positive
        
        return depth_planes


# Demonstrate the difference
torch.manual_seed(42)

# Simulated mono depth for a scene
H, W = 64, 64
y, x = torch.meshgrid(torch.linspace(0, 1, H), torch.linspace(0, 1, W), indexing='ij')
mono_depth = 2.0 + 3.0 * x + 1.5 * torch.exp(-((x-0.5)**2 + (y-0.5)**2) / 0.1)
mono_depth = mono_depth.unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]

# Fixed sampling (MVSplat style)
fixed_planes = torch.linspace(0.5, 10.0, 32).view(1, -1, 1, 1).expand(1, -1, H, W)

# Adaptive sampling (DepthSplat style)
sampler = DepthGuidedPlaneSampler(num_planes=32, spread=0.3)
adaptive_planes = sampler(mono_depth)

print(f"Fixed planes range: [{fixed_planes.min():.1f}, {fixed_planes.max():.1f}]")
print(f"Adaptive planes range: [{adaptive_planes.min():.1f}, {adaptive_planes.max():.1f}]")
print(f"Mono depth range: [{mono_depth.min():.1f}, {mono_depth.max():.1f}]")

In [ ]:
# Visualize fixed vs adaptive plane sampling

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Mono depth
ax = axes[0]
im = ax.imshow(mono_depth[0, 0].numpy(), cmap='plasma')
ax.set_title('Monocular Depth Prior', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046)
ax.axis('off')

# Fixed vs Adaptive at two pixel locations
pixels = [(16, 16), (48, 48)]  # Near (shallow depth) and far (deep depth)
pixel_labels = ['Near pixel (16,16)', 'Far pixel (48,48)']
colors_px = ['#2196F3', '#FF5722']

# Mark pixels on depth map
for (py, px), color in zip(pixels, colors_px):
    ax.plot(px, py, 'o', color=color, markersize=10, markeredgecolor='white', lw=2)

# Fixed planes
ax = axes[1]
for i, ((py, px), label, color) in enumerate(zip(pixels, pixel_labels, colors_px)):
    fp = fixed_planes[0, :, py, px].numpy()
    gt_d = mono_depth[0, 0, py, px].item()
    ax.scatter(fp, [i]*len(fp), c=color, s=30, alpha=0.6, label=label)
    ax.axvline(gt_d, color=color, linestyle='--', alpha=0.5, lw=2)
ax.set_xlabel('Depth', fontsize=11)
ax.set_title('Fixed Planes (MVSplat)\nMany wasted hypotheses', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_yticks([])
ax.grid(True, alpha=0.3, axis='x')

# Adaptive planes
ax = axes[2]
for i, ((py, px), label, color) in enumerate(zip(pixels, pixel_labels, colors_px)):
    ap = adaptive_planes[0, :, py, px].numpy()
    gt_d = mono_depth[0, 0, py, px].item()
    ax.scatter(ap, [i]*len(ap), c=color, s=30, alpha=0.6, label=label)
    ax.axvline(gt_d, color=color, linestyle='--', alpha=0.5, lw=2)
ax.set_xlabel('Depth', fontsize=11)
ax.set_title('Adaptive Planes (DepthSplat)\nFocused around prior', fontsize=12,
             fontweight='bold', color='#4CAF50')
ax.legend(fontsize=9)
ax.set_yticks([])
ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Depth Plane Sampling: Fixed vs Adaptive',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.3 DepthSplat Integration Architecture

DepthSplat integrates the depth prior at multiple levels:

1. **Feature Fusion**: Depth features concatenated with image features
2. **Plane Sampling**: Adaptive depth hypotheses around monocular prediction
3. **Cost Volume Conditioning**: Depth prior modulates cost volume weights
4. **Depth Refinement**: Final depth = prior + learned residual

In [ ]:
class SimplifiedDepthSplat(nn.Module):
    """
    Simplified DepthSplat architecture for educational purposes.
    
    Key differences from MVSplat:
    1. Uses monocular depth prior (from pretrained model)
    2. Adaptive depth plane sampling around prior
    3. Depth features fused with image features
    4. Final depth = prior + learned residual
    """
    def __init__(self, feature_dim=32, num_depth_planes=16):
        super().__init__()
        
        # 1. Image feature encoder (shared across views)
        self.image_encoder = nn.Sequential(
            nn.Conv2d(3, feature_dim, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(feature_dim, feature_dim, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        
        # 2. Depth prior encoder (simulates DPT / Depth Anything)
        self.depth_prior = DepthPriorEncoder(3, feature_dim)
        
        # 3. Feature fusion (image + depth features)
        self.fusion = nn.Sequential(
            nn.Conv2d(feature_dim * 2, feature_dim, 1),
            nn.ReLU(inplace=True),
        )
        
        # 4. Adaptive plane sampler
        self.plane_sampler = DepthGuidedPlaneSampler(
            num_planes=num_depth_planes, spread=0.3
        )
        
        # 5. Cost volume processing (simplified 3D CNN)
        self.cost_net = nn.Sequential(
            nn.Conv3d(feature_dim, feature_dim, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(feature_dim, 1, 3, padding=1),
        )
        
        # 6. Depth residual prediction
        self.depth_residual = nn.Conv2d(feature_dim + 1, 1, 1)
        
        # 7. Gaussian parameter heads
        self.scale_head = nn.Conv2d(feature_dim, 3, 1)
        self.rotation_head = nn.Conv2d(feature_dim, 4, 1)
        self.opacity_head = nn.Conv2d(feature_dim, 1, 1)
        
        self.num_depth_planes = num_depth_planes
    
    def forward(self, images, K):
        """
        Args:
            images: [B, N, 3, H, W] multi-view images (N=2 typically)
            K: [B, 3, 3] intrinsics
        Returns:
            dict with Gaussian parameters
        """
        B, N, C, H, W = images.shape
        
        # Process each view
        all_depths = []
        all_features = []
        
        for v in range(N):
            img = images[:, v]  # [B, 3, H, W]
            
            # Step 1: Extract image features
            img_feat = self.image_encoder(img)  # [B, F, H, W]
            
            # Step 2: Get depth prior
            mono_depth, depth_feat = self.depth_prior(img)  # [B, 1, H, W], [B, F, H, W]
            
            # Step 3: Fuse image and depth features
            fused = self.fusion(torch.cat([img_feat, depth_feat], dim=1))  # [B, F, H, W]
            
            # Step 4: Adaptive depth plane sampling
            depth_planes = self.plane_sampler(mono_depth)  # [B, D, H, W]
            
            # Step 5: Build cost volume (simplified - just expand features)
            # In real implementation: warp source features to each depth plane
            cost_vol = fused.unsqueeze(2).expand(-1, -1, self.num_depth_planes, -1, -1)
            
            # Step 6: Process cost volume
            cost_probs = self.cost_net(cost_vol)  # [B, 1, D, H, W]
            cost_probs = F.softmax(cost_probs.squeeze(1), dim=1)  # [B, D, H, W]
            
            # Step 7: Soft argmin over adaptive planes
            mvs_depth = (cost_probs * depth_planes).sum(dim=1, keepdim=True)  # [B, 1, H, W]
            
            # Step 8: Depth residual refinement
            residual_input = torch.cat([fused, mono_depth], dim=1)
            depth_residual = self.depth_residual(residual_input)  # [B, 1, H, W]
            final_depth = mvs_depth + 0.1 * depth_residual  # Prior + residual
            
            all_depths.append(final_depth)
            all_features.append(fused)
        
        # Predict Gaussian parameters from last fused features
        feat = all_features[0]
        depth = all_depths[0]
        
        scales = torch.exp(self.scale_head(feat).clamp(-5, 3))
        rotations = F.normalize(self.rotation_head(feat), dim=1)
        opacities = torch.sigmoid(self.opacity_head(feat))
        
        return {
            'depth': depth,               # [B, 1, H, W]
            'scales': scales,              # [B, 3, H, W]
            'rotations': rotations,        # [B, 4, H, W]
            'opacities': opacities,        # [B, 1, H, W]
            'colors': images[:, 0],        # [B, 3, H, W] use input color
            'mono_depth': mono_depth,      # [B, 1, H, W] for visualization
        }


# Test the model
torch.manual_seed(42)
model = SimplifiedDepthSplat(feature_dim=32, num_depth_planes=16)
model.eval()

# Synthetic input
images = torch.randn(1, 2, 3, 64, 64)  # 2 views
K = torch.tensor([[50, 0, 32], [0, 50, 32], [0, 0, 1]], dtype=torch.float32).unsqueeze(0)

with torch.no_grad():
    output = model(images, K)

print("SimplifiedDepthSplat Output:")
for key, val in output.items():
    print(f"  {key:12s}: {list(val.shape)}")

n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {n_params:,}")

In [ ]:
# Visualize the DepthSplat architecture

fig, ax = plt.subplots(1, 1, figsize=(18, 12))
ax.set_xlim(0, 18)
ax.set_ylim(0, 13)
ax.axis('off')
ax.set_title('DepthSplat Architecture Overview', fontsize=16, fontweight='bold', pad=15)

def draw_box(ax, x, y, w, h, text, color, fontsize=9, subtext=None, bold=True):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='black', lw=1.5)
    ax.add_patch(box)
    dy = 0.12 if subtext else 0
    weight = 'bold' if bold else 'normal'
    ax.text(x + w/2, y + h/2 + dy, text,
            ha='center', va='center', fontsize=fontsize, fontweight=weight)
    if subtext:
        ax.text(x + w/2, y + h/2 - 0.2, subtext,
                ha='center', va='center', fontsize=7, style='italic', color='#444')

def draw_arrow(ax, x1, y1, x2, y2, color='#666'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

# Input
draw_box(ax, 1, 11.5, 3, 0.8, 'Input Images', '#E0E0E0', subtext='[B, N, 3, H, W]')

# Two branches
# Left: Image Encoder
draw_box(ax, 0.5, 9.8, 3, 0.8, 'Image Encoder', '#BBDEFB', subtext='CNN / ViT backbone')
draw_arrow(ax, 2.5, 11.5, 2, 10.6)

# Right: Depth Prior
draw_box(ax, 5, 9.8, 3.5, 0.8, 'Depth Prior Network', '#FFE0B2',
         subtext='DPT / Depth Anything V2')
draw_arrow(ax, 2.5, 11.5, 6.75, 10.6)

# NEW tag on depth prior
ax.text(8.7, 10.4, 'NEW', fontsize=8, fontweight='bold', color='white',
        bbox=dict(boxstyle='round', facecolor='#F44336', edgecolor='none'))

# Feature Fusion
draw_box(ax, 2, 8.0, 3.5, 0.8, 'Feature Fusion', '#C8E6C9',
         subtext='Concat + Conv1x1')
draw_arrow(ax, 2, 9.8, 3.75, 8.8)
draw_arrow(ax, 6.75, 9.8, 3.75, 8.8)

# Depth Prior also feeds to Plane Sampler
draw_box(ax, 7, 8.0, 3.5, 0.8, 'Adaptive Plane Sampler', '#FFE0B2',
         subtext='Depth-guided hypotheses')
draw_arrow(ax, 6.75, 9.8, 8.75, 8.8)
ax.text(10.7, 8.4, 'NEW', fontsize=8, fontweight='bold', color='white',
        bbox=dict(boxstyle='round', facecolor='#F44336', edgecolor='none'))

# Cost Volume
draw_box(ax, 2, 6.2, 9, 1.0, 'Depth-conditioned Cost Volume', '#BBDEFB',
         subtext='Warp features to adaptive depth planes + 3D U-Net')
draw_arrow(ax, 3.75, 8.0, 6.5, 7.2)
draw_arrow(ax, 8.75, 8.0, 6.5, 7.2)

# Depth Refinement
draw_box(ax, 2, 4.5, 4, 0.8, 'Depth Refinement', '#C8E6C9',
         subtext='Prior + Residual')
draw_arrow(ax, 6.5, 6.2, 4, 5.3)
ax.text(6.2, 5.0, 'NEW', fontsize=8, fontweight='bold', color='white',
        bbox=dict(boxstyle='round', facecolor='#F44336', edgecolor='none'))

# Gaussian Heads
draw_box(ax, 7, 4.5, 4, 0.8, 'Gaussian Heads', '#E1BEE7',
         subtext='scale, rotation, opacity')
draw_arrow(ax, 6.5, 6.2, 9, 5.3)

# Pixel-aligned Gaussians
draw_box(ax, 3, 2.8, 7, 0.8, 'Pixel-aligned Gaussians', '#FFF9C4',
         subtext='Back-project refined depth to 3D + merge views')
draw_arrow(ax, 4, 4.5, 6.5, 3.6)
draw_arrow(ax, 9, 4.5, 6.5, 3.6)

# Rendering
draw_box(ax, 4, 1.2, 5, 0.8, 'Differentiable Rendering', '#FFCDD2',
         subtext='gsplat / diff-gaussian-rasterization')
draw_arrow(ax, 6.5, 2.8, 6.5, 2.0)

# Right panel: key improvements
improvements = [
    'DepthSplat vs MVSplat:',
    '',
    '1. Monocular depth prior',
    '   -> Better depth initialization',
    '   -> Works on textureless regions',
    '',
    '2. Adaptive plane sampling',
    '   -> No wasted depth hypotheses',
    '   -> Better depth resolution',
    '',
    '3. Feature fusion',
    '   -> Depth-aware features',
    '   -> Geometry + appearance',
    '',
    '4. Depth refinement',
    '   -> Prior + learned residual',
    '   -> Multi-view consistent',
    '',
    'Result: +0.86 dB PSNR',
    '        on RE10K benchmark',
]

for i, line in enumerate(improvements):
    weight = 'bold' if i == 0 or 'Result' in line else 'normal'
    color = '#4CAF50' if 'Result' in line or 'RE10K' in line else 'black'
    ax.text(12.5, 12 - i * 0.52, line, fontsize=8, fontweight=weight,
            fontfamily='monospace', color=color)

plt.tight_layout()
plt.show()

## 3. Depth Anything: The Foundation

### 3.1 How Depth Anything Works

Depth Anything is the most popular depth prior used in DepthSplat:

```
Input Image (H x W x 3)
    ↓
DINOv2 ViT Backbone (pretrained)
    ↓ 
Multi-scale features
    ↓
DPT Decoder (Feature Pyramid)
    ↓
Relative Depth Map (H x W)
```

### 3.2 Depth Anything V2 Improvements

| Aspect | V1 | V2 |
|--------|----|---------|
| Training data | 62M real unlabeled | 595K synthetic + 62M unlabeled |
| Backbone | DINOv2 ViT-L | DINOv2 ViT-L/G |
| Pseudo labels | MiDaS teacher | Synthetic-trained teacher |
| Metric depth | No | Yes (fine-tuned version) |
| Edge quality | Good | Better (synthetic supervision) |

### 3.3 Scale and Shift Alignment

Before using monocular depth with MVS, we need to align scale and shift:

In [ ]:
def align_depth_scale_shift(mono_depth, mvs_depth, mask=None):
    """
    Align monocular depth to MVS depth via least-squares.
    
    Solves: mvs_depth = scale * mono_depth + shift
    
    This is essential because monocular depth has:
    - Unknown scale (relative, not metric)
    - Unknown shift (offset is arbitrary)
    """
    if mask is None:
        mask = torch.ones_like(mono_depth, dtype=torch.bool)
    
    mono_flat = mono_depth[mask].float()
    mvs_flat = mvs_depth[mask].float()
    
    # Least squares: [scale, shift] = (A^T A)^{-1} A^T b
    A = torch.stack([mono_flat, torch.ones_like(mono_flat)], dim=1)  # [N, 2]
    b = mvs_flat  # [N]
    
    # Normal equations
    ATA = A.T @ A  # [2, 2]
    ATb = A.T @ b  # [2]
    
    params = torch.linalg.solve(ATA, ATb)  # [scale, shift]
    scale, shift = params[0].item(), params[1].item()
    
    aligned = scale * mono_depth + shift
    return aligned, scale, shift


# Demonstrate scale-shift alignment
torch.manual_seed(42)
H, W = 64, 64

# Ground truth depth
y, x = torch.meshgrid(torch.linspace(0, 1, H), torch.linspace(0, 1, W), indexing='ij')
gt_depth = 2.0 + 3.0 * x + torch.randn(H, W) * 0.05

# Monocular depth (different scale and shift)
mono_raw = 0.3 * gt_depth - 0.5 + torch.randn(H, W) * 0.02  # Different scale + shift

# MVS depth (sparse but correct scale)
mvs_sparse = gt_depth + torch.randn(H, W) * 0.2

# Align
aligned, scale, shift = align_depth_scale_shift(mono_raw, mvs_sparse)

print(f"Alignment: scale={scale:.3f}, shift={shift:.3f}")
print(f"Error before alignment: {(mono_raw - gt_depth).abs().mean():.3f}")
print(f"Error after alignment:  {(aligned - gt_depth).abs().mean():.3f}")
print(f"MVS error:              {(mvs_sparse - gt_depth).abs().mean():.3f}")

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, data, title in zip(axes,
    [gt_depth.numpy(), mono_raw.numpy(), aligned.numpy(), mvs_sparse.numpy()],
    ['Ground Truth', 'Mono (raw)', 'Mono (aligned)', 'MVS (noisy)']):
    im = ax.imshow(data, cmap='plasma')
    ax.set_title(title, fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.axis('off')

plt.suptitle('Scale-Shift Alignment of Monocular Depth', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Single-Image 3D: Splatt3R & Flash3D

### 4.1 The Single-Image Challenge

MVSplat and pixelSplat require **at least 2 input views**. But what if we only have **one image**?

Several 2024-2025 methods tackle single-image 3D Gaussian prediction:

| Method | Input | Output | Key Innovation |
|--------|-------|--------|----------------|
| **Splatt3R** | 1 image | 3D Gaussians | DUSt3R backbone + Gaussian heads |
| **Flash3D** | 1 image | 3D Gaussians | Efficient single-view with depth prior |
| **LGM** | 1-4 images | 3D Gaussians | Multi-view diffusion + Gaussian regression |
| **GRM** | 4 sparse views | 3D Gaussians | Large reconstruction model |

### 4.2 Splatt3R Architecture

```
Single Input Image
    ↓
DUSt3R Encoder (ViT)
    ↓
Pointmap Prediction (3D structure)
    ↓
Gaussian Parameter Heads
  ├── Position (from pointmap)
  ├── Scale (learned)
  ├── Rotation (learned)
  ├── Opacity (learned)
  └── Color/SH (learned)
    ↓
Complete 3D Gaussian Scene
```

### 4.3 Challenges of Single-Image 3D

1. **Depth ambiguity**: No multi-view cue, must rely entirely on learned priors
2. **Occluded regions**: Cannot see behind objects from a single view
3. **Scale ambiguity**: Absolute scale is fundamentally undetermined
4. **Hallucination**: Model must "imagine" unseen regions

In [ ]:
# Visualize single-image 3D challenge and methods

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Top row: Method comparison
methods_data = [
    ('MVSplat / pixelSplat', '2+ views', 'Multi-view cues',
     ['Multi-view correspondence', 'Cost volume / attention', 'Explicit geometry'],
     '#BBDEFB'),
    ('Splatt3R / Flash3D', '1 view', 'Learned priors',
     ['DUSt3R/depth backbone', 'Pointmap prediction', 'No multi-view needed'],
     '#C8E6C9'),
    ('LGM / GRM', '1-4 views', 'Generative + recon',
     ['Multi-view diffusion', 'Large-scale pretraining', 'Object-centric'],
     '#FFE0B2'),
]

for i, (name, input_type, approach, details, color) in enumerate(methods_data):
    ax = axes[0, i]
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_facecolor(color)
    
    ax.text(0.5, 0.9, name, ha='center', va='center', fontsize=12, fontweight='bold')
    ax.text(0.5, 0.78, f'Input: {input_type}', ha='center', fontsize=10,
            style='italic', color='#666')
    ax.text(0.5, 0.66, f'Approach: {approach}', ha='center', fontsize=10)
    
    for j, detail in enumerate(details):
        ax.text(0.1, 0.48 - j*0.12, f'  {detail}', fontsize=9)
    
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor('black'); spine.set_linewidth(1.5)

# Bottom row: Quality vs Input views trade-off
ax = axes[1, 0]
n_views = [1, 1, 2, 2, 4, 'N']
quality = [22, 23, 25.9, 26.0, 27, 28.5]
method_names = ['Flash3D', 'Splatt3R', 'pixelSplat', 'MVSplat', 'GRM', 'Per-scene']
colors = ['#4CAF50', '#4CAF50', '#FF9800', '#2196F3', '#FF5722', '#9E9E9E']

for i, (nv, q, mn, c) in enumerate(zip(n_views, quality, method_names, colors)):
    x_pos = i
    ax.bar(x_pos, q, color=c, alpha=0.8, edgecolor='black', lw=0.5)
    ax.text(x_pos, q + 0.3, f'{q}', ha='center', fontsize=8)
    ax.text(x_pos, 20.8, f'{nv}v', ha='center', fontsize=8, color='#666')

ax.set_xticks(range(len(method_names)))
ax.set_xticklabels(method_names, fontsize=8, rotation=15)
ax.set_ylabel('PSNR (dB)')
ax.set_title('Quality vs Number of Input Views', fontsize=11, fontweight='bold')
ax.set_ylim(20, 30)
ax.grid(True, alpha=0.3, axis='y')

# Single-image challenges
ax = axes[1, 1]
challenges = [
    ('Depth\nAmbiguity', 0.85, 'No stereo cues'),
    ('Occlusion', 0.7, 'Cannot see behind'),
    ('Scale', 0.6, 'No metric reference'),
    ('Hallucination', 0.5, 'Must "imagine" unseen'),
]
y_pos = [0.8, 0.6, 0.4, 0.2]
for (name, severity, desc), yp in zip(challenges, y_pos):
    ax.barh(yp, severity, height=0.12, color='#F44336', alpha=severity, edgecolor='black')
    ax.text(0.02, yp, f'{name}', va='center', fontsize=9, fontweight='bold')
    ax.text(severity + 0.02, yp, desc, va='center', fontsize=8, color='#666')

ax.set_xlim(0, 1.2); ax.set_ylim(0, 1)
ax.set_xlabel('Challenge Severity')
ax.set_title('Single-Image 3D Challenges', fontsize=11, fontweight='bold')
ax.set_yticks([])
ax.grid(True, alpha=0.3, axis='x')

# Timeline
ax = axes[1, 2]
timeline = [
    (2024.0, 'pixelSplat (CVPR 2024)', '#FF9800'),
    (2024.3, 'MVSplat (ECCV 2024)', '#2196F3'),
    (2024.5, 'LGM', '#9C27B0'),
    (2024.7, 'Flash3D', '#4CAF50'),
    (2024.9, 'Splatt3R', '#4CAF50'),
    (2025.0, 'DepthSplat', '#F44336'),
    (2025.3, 'Multi-view scaling', '#E91E63'),
]
for t, name, color in timeline:
    ax.scatter(t, 0.5, s=100, color=color, zorder=5, edgecolor='black')
    offset = 0.15 if timeline.index((t, name, color)) % 2 == 0 else -0.15
    ax.text(t, 0.5 + offset, name, ha='center', fontsize=8, rotation=30)

ax.axhline(0.5, color='gray', lw=1, alpha=0.5)
ax.set_xlim(2023.8, 2025.8)
ax.set_ylim(0, 1)
ax.set_xlabel('Year')
ax.set_title('Feed-forward 3DGS Timeline', fontsize=11, fontweight='bold')
ax.set_yticks([])

plt.suptitle('Single-Image 3D and Method Evolution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Multi-View Scaling: From 2 to N Views

### 5.1 The Scaling Challenge

MVSplat and pixelSplat are designed for **2 input views**. Scaling to N views introduces:

| Challenge | 2 Views | N Views |
|-----------|---------|----------|
| Cost Volume pairs | 1 pair | O(N^2) or O(N) pairs |
| Memory | Fixed | Grows with N |
| Computation | Fixed | Grows with N |
| View selection | Trivial | Need strategy |
| Merging | Simple concat | Need attention/fusion |

### 5.2 Approaches to Multi-View Scaling

1. **Reference-based**: Pick 1 reference, compute cost volume with all sources
2. **Pairwise pooling**: Compute all pairs, pool results
3. **Hierarchical**: Group views, process in stages
4. **Attention-based**: Use transformer attention across all views (VGGT approach)

In [ ]:
# Analyze scaling complexity

def compute_complexity(n_views, method='pairwise'):
    """Compute relative complexity for different multi-view strategies."""
    if method == 'pairwise':
        return n_views * (n_views - 1) / 2  # All pairs
    elif method == 'reference':
        return n_views - 1  # Reference vs each source
    elif method == 'hierarchical':
        import math
        return n_views * math.log2(max(n_views, 2))  # Log-depth tree
    elif method == 'attention':
        return n_views ** 2  # Full attention


n_views_range = np.arange(2, 33)
methods = ['pairwise', 'reference', 'hierarchical', 'attention']
labels = ['Pairwise O(N\u00b2)', 'Reference O(N)', 'Hierarchical O(N log N)', 'Full Attention O(N\u00b2)']
colors = ['#F44336', '#4CAF50', '#FF9800', '#2196F3']

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Complexity scaling
ax = axes[0]
for method, label, color in zip(methods, labels, colors):
    complexity = [compute_complexity(n, method) for n in n_views_range]
    ax.plot(n_views_range, complexity, '-o', label=label, color=color,
            lw=2, markersize=3)

ax.set_xlabel('Number of Input Views', fontsize=11)
ax.set_ylabel('Relative Computation', fontsize=11)
ax.set_title('Multi-View Scaling Complexity', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(2, 32)

# Quality vs views
ax = axes[1]
views_quality = [2, 4, 8, 16, 32]
# Simulated quality improvement with more views
quality_mvsplat = [25.97, 26.5, 27.0, 27.3, 27.4]  # Diminishing returns
quality_attention = [25.5, 26.8, 27.5, 28.2, 28.8]  # Better scaling

ax.plot(views_quality, quality_mvsplat, 'o-', color='#2196F3', lw=2,
        markersize=8, label='MVSplat-style (Cost Volume)')
ax.plot(views_quality, quality_attention, 's-', color='#9C27B0', lw=2,
        markersize=8, label='Attention-based (VGGT-style)')

ax.axhline(28.5, color='gray', linestyle='--', alpha=0.5, label='Per-scene 3DGS')
ax.set_xlabel('Number of Input Views', fontsize=11)
ax.set_ylabel('PSNR (dB)', fontsize=11)
ax.set_title('Quality Scaling with More Views', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xscale('log', base=2)
ax.set_xticks(views_quality)
ax.set_xticklabels(views_quality)

plt.suptitle('Multi-View Scaling: Challenges and Trade-offs',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.3 Connection to VGGT (Phase 5 Preview)

**VGGT (Visual Geometry Grounded Transformer)** represents the next paradigm:

```
N Input Images
    ↓
ViT Backbone (shared)
    ↓
Cross-view Transformer (full attention over all views)
    ↓
Multi-head Predictions:
  ├── Camera poses (no SfM needed!)
  ├── Depth maps
  ├── Point clouds
  └── 3D Gaussians
```

**Key advantage**: VGGT handles N views natively with transformer attention, doesn't need Cost Volume or epipolar geometry.

In [ ]:
# Compare paradigm evolution

paradigms = [
    {
        'name': 'Per-scene 3DGS\n(Phase 1)',
        'pros': ['Highest quality', 'Well understood', 'Flexible'],
        'cons': ['Slow (30+ min)', 'No generalization', 'Needs COLMAP'],
        'color': '#E0E0E0',
    },
    {
        'name': 'MVSplat/pixelSplat\n(Phase 4 Core)',
        'pros': ['Fast (~25ms)', 'Cross-scene', '2 views enough'],
        'cons': ['Lower quality', 'Fixed 2 views', 'Needs poses'],
        'color': '#BBDEFB',
    },
    {
        'name': 'DepthSplat\n(Depth Prior)',
        'pros': ['Better quality', 'Depth-guided', 'Handles textureless'],
        'cons': ['Extra computation', 'Depth model needed', 'Still 2 views'],
        'color': '#C8E6C9',
    },
    {
        'name': 'VGGT\n(Phase 5)',
        'pros': ['N views native', 'No poses needed', 'Multi-task'],
        'cons': ['Heavy compute', 'New architecture', 'Less mature'],
        'color': '#E1BEE7',
    },
]

fig, axes = plt.subplots(1, 4, figsize=(20, 6))

for ax, p in zip(axes, paradigms):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_facecolor(p['color'])
    
    ax.text(0.5, 0.92, p['name'], ha='center', va='center',
            fontsize=11, fontweight='bold')
    
    ax.text(0.05, 0.72, 'Strengths:', fontsize=9, fontweight='bold', color='#2E7D32')
    for j, pro in enumerate(p['pros']):
        ax.text(0.08, 0.63 - j*0.09, f'+ {pro}', fontsize=8, color='#2E7D32')
    
    ax.text(0.05, 0.32, 'Weaknesses:', fontsize=9, fontweight='bold', color='#C62828')
    for j, con in enumerate(p['cons']):
        ax.text(0.08, 0.23 - j*0.09, f'- {con}', fontsize=8, color='#C62828')
    
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor('black'); spine.set_linewidth(1.5)

# Add evolution arrows
fig.patches.extend([
    FancyArrowPatch((0.27, 0.5), (0.30, 0.5), transform=fig.transFigure,
                    arrowstyle='->', mutation_scale=20, lw=2, color='gray'),
    FancyArrowPatch((0.50, 0.5), (0.53, 0.5), transform=fig.transFigure,
                    arrowstyle='->', mutation_scale=20, lw=2, color='gray'),
    FancyArrowPatch((0.73, 0.5), (0.76, 0.5), transform=fig.transFigure,
                    arrowstyle='->', mutation_scale=20, lw=2, color='gray'),
])

plt.suptitle('Evolution of 3D Gaussian Splatting Paradigms',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. 2025 Trends and Future Directions

### 6.1 Video-based Feed-forward 3DGS

Instead of sparse image pairs, use video sequences:

- **Temporal consistency**: Enforce smooth Gaussian transitions across frames
- **Streaming reconstruction**: Process frames incrementally
- **Connection to SLAM**: Feed-forward SLAM = predict Gaussians per frame

### 6.2 Dynamic Scene Handling

- **4D Gaussian Splatting**: Time-varying Gaussian parameters
- **Motion decomposition**: Static background + dynamic foreground
- **Deformation fields**: Canonical space + per-frame deformation

### 6.3 Language-Guided 3D

- **Text-to-3D**: Generate Gaussians from text descriptions
- **Open-vocabulary segmentation**: Language features in Gaussian representation
- **LLM + 3DGS**: Spatial reasoning with language models

### 6.4 Efficiency and Deployment

- **Model compression**: Pruning, quantization, distillation
- **Mobile deployment**: On-device feed-forward 3DGS
- **Streaming**: Progressive Gaussian transmission

In [ ]:
# Visualize 2025 research landscape

fig, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.set_xlim(0, 16); ax.set_ylim(0, 11)
ax.axis('off')
ax.set_title('2025 Feed-forward 3DGS Research Landscape',
             fontsize=16, fontweight='bold', pad=15)

# Central node
center_x, center_y = 8, 5.5
box = FancyBboxPatch((center_x-1.5, center_y-0.4), 3, 0.8,
                     boxstyle='round,pad=0.15', facecolor='#2196F3',
                     edgecolor='black', lw=2)
ax.add_patch(box)
ax.text(center_x, center_y, 'Feed-forward\n3DGS', ha='center', va='center',
        fontsize=12, fontweight='bold', color='white')

# Branches
branches = [
    # (angle_approx, x, y, title, items, color)
    (8, 9.5, 'Depth Priors', ['DepthSplat', 'Depth Anything V2', 'Metric3D v2'], '#4CAF50'),
    (13, 9.5, 'Single-Image 3D', ['Splatt3R', 'Flash3D', 'LGM'], '#FF9800'),
    (14, 7.5, 'Multi-View Scaling', ['VGGT', 'Transformer attention', 'N-view native'], '#9C27B0'),
    (13.5, 5.5, 'Dynamic Scenes', ['4D Gaussians', 'Deformation fields', 'Motion decomp'], '#F44336'),
    (8, 3.5, 'Video-based', ['Temporal coherence', 'Streaming recon', 'Feed-forward SLAM'], '#00BCD4'),
    (3, 3.5, 'Language-Guided', ['Text-to-3D', 'Open-vocab 3D', 'LLM + 3DGS'], '#E91E63'),
    (2, 5.5, 'Efficiency', ['Mobile deployment', 'Model compression', 'Streaming'], '#795548'),
    (2.5, 7.5, 'Foundation Models', ['DUSt3R backbone', 'Large-scale pretrain', 'Multi-task'], '#607D8B'),
]

for bx, by, title, items, color in branches:
    # Draw branch box
    box_w, box_h = 2.8, 1.4
    box = FancyBboxPatch((bx - box_w/2, by - box_h/2), box_w, box_h,
                         boxstyle='round,pad=0.1', facecolor=color, alpha=0.2,
                         edgecolor=color, lw=1.5)
    ax.add_patch(box)
    
    # Title
    ax.text(bx, by + box_h/2 - 0.2, title, ha='center', va='center',
            fontsize=9, fontweight='bold', color=color)
    
    # Items
    for j, item in enumerate(items):
        ax.text(bx, by + 0.05 - j*0.3, f'  {item}', ha='center', va='center',
                fontsize=7.5)
    
    # Arrow from center
    ax.annotate('', xy=(bx, by), xytext=(center_x, center_y),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5, alpha=0.6))

plt.tight_layout()
plt.show()

In [ ]:
# Comprehensive benchmark table

print("="*80)
print("   2024-2025 Feed-forward 3DGS Methods: Comprehensive Comparison")
print("="*80)

headers = ['Method', 'Year', 'Views', 'PSNR', 'Speed', 'Depth Prior', 'Poses']
rows = [
    ['pixelSplat', '2024', '2', '25.89', '~10 FPS', 'No', 'Required'],
    ['MVSplat', '2024', '2', '25.97', '~22 FPS', 'No', 'Required'],
    ['Flash3D', '2024', '1', '~23', '~30 FPS', 'Yes', 'Not needed'],
    ['Splatt3R', '2024', '1', '~23.5', '~15 FPS', 'DUSt3R', 'Not needed'],
    ['LGM', '2024', '1-4', '~24', '~5 FPS', 'No', 'Required'],
    ['DepthSplat', '2025', '2', '26.83', '~15 FPS', 'Yes (DA V2)', 'Required'],
    ['VGGT', '2025', 'N', '~27', '~3 FPS', 'Implicit', 'Predicted'],
    ['Per-scene 3DGS', '2023', 'N', '28.5+', '30 min', 'No', 'SfM needed'],
]

# Print table
col_widths = [15, 6, 6, 7, 10, 12, 12]
header_line = ' | '.join(h.ljust(w) for h, w in zip(headers, col_widths))
print(f"\n{header_line}")
print('-' * len(header_line))

for row in rows:
    line = ' | '.join(val.ljust(w) for val, w in zip(row, col_widths))
    print(line)

print(f"\n{'='*80}")
print("\nKey Takeaways:")
print("1. DepthSplat shows depth priors consistently improve quality (+0.86 dB)")
print("2. Single-image methods are viable but ~3 dB below multi-view")
print("3. Attention-based methods (VGGT) scale better to many views")
print("4. Speed-quality trade-off: fastest methods sacrifice ~2-3 dB")
print("5. Feed-forward gap vs per-scene narrowing: was 5+ dB, now ~2 dB")

In [ ]:
# Summary

summary = """
=====================================================================
   Notebook 08 Summary: DepthSplat & 2025 Advances
=====================================================================

1. DEPTH PRIORS
   - Monocular depth (Depth Anything V2) provides strong geometric prior
   - Scale-shift alignment needed for integration with MVS
   - DepthSplat: +0.86 dB PSNR over MVSplat on RE10K

2. DEPTHSPLAT ARCHITECTURE
   - Depth-guided adaptive plane sampling (no wasted hypotheses)
   - Feature fusion (image + depth features)
   - Depth refinement (prior + learned residual)

3. SINGLE-IMAGE 3D
   - Splatt3R: DUSt3R backbone + Gaussian heads
   - Flash3D: Efficient single-view with depth prior
   - Challenges: depth ambiguity, occlusion, hallucination

4. MULTI-VIEW SCALING
   - Cost Volume: O(N) with reference-based, O(N^2) pairwise
   - Attention-based: Better scaling with more views
   - VGGT: Native N-view processing (Phase 5)

5. 2025 TRENDS
   - Video-based feed-forward 3DGS
   - Dynamic scene handling (4D Gaussians)
   - Language-guided 3D reconstruction
   - Mobile deployment and efficiency

=====================================================================
"""
print(summary)

## What's Next?

**[09_mvsplat_vs_pixelsplat_comparison.ipynb](./09_mvsplat_vs_pixelsplat_comparison.ipynb)** - Systematic experimental comparison of MVSplat vs pixelSplat: architecture, performance, and when to use each method.

---

## References

1. DepthSplat: https://arxiv.org/abs/2412.18010
2. Depth Anything V2: https://arxiv.org/abs/2406.09414
3. Splatt3R: https://arxiv.org/abs/2408.07648
4. Flash3D: https://arxiv.org/abs/2406.04343
5. LGM: https://arxiv.org/abs/2402.05054
6. VGGT: https://arxiv.org/abs/2503.11651
7. DPT: https://arxiv.org/abs/2103.13413
8. MiDaS: https://arxiv.org/abs/1907.01341